# Inference

Load a saved model version from `model/model_<n>/` together with its `tokenizer.json`, then generate text with the same tokenizer used during training.

In [20]:
from pathlib import Path
import importlib
import torch

import model as model_module
import train_tokenizer as tokenizer_module

importlib.reload(model_module)
importlib.reload(tokenizer_module)

from model import SimpleGPTPredictor, device
from train_tokenizer import BPETokenizer

EMBED_SIZE = 32
NUM_HEADS = 4
MAX_LEN = 100
MODEL_ROOT = Path("model")
MODEL_VERSION = 15  # Set to an int like 3 to load model/model_3


In [21]:
def list_model_versions(model_root: Path) -> list[int]:
    versions = []
    for path in model_root.glob("model_*"):
        if not path.is_dir():
            continue
        suffix = path.name.removeprefix("model_")
        if suffix.isdigit():
            versions.append(int(suffix))
    return sorted(versions)


def resolve_model_dir(model_root: Path, version: int | None) -> Path:
    versions = list_model_versions(model_root)
    if not versions:
        raise FileNotFoundError(f"No model versions found under {model_root}")
    resolved = versions[-1] if version is None else version
    model_dir = model_root / f"model_{resolved}"
    if not model_dir.is_dir():
        raise FileNotFoundError(f"Model directory not found: {model_dir}")
    return model_dir


def load_state_dict(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)


def infer_num_layers(state_dict, prefix="encoder.layers.") -> int:
    layer_ids = {int(key.split(".")[2]) for key in state_dict if key.startswith(prefix)}
    return (max(layer_ids) + 1) if layer_ids else 0


MODEL_DIR = resolve_model_dir(MODEL_ROOT, MODEL_VERSION)
MODEL_PATH = MODEL_DIR / "model.pth"
TOKENIZER_PATH = MODEL_DIR / "tokenizer.json"

tokenizer = BPETokenizer.load(str(TOKENIZER_PATH))
state_dict = load_state_dict(MODEL_PATH)
num_layers = infer_num_layers(state_dict)

model = SimpleGPTPredictor(
    vocab_size=tokenizer.vocab_size,
    embed_size=EMBED_SIZE,
    num_heads=NUM_HEADS,
    max_len=MAX_LEN,
    num_layers=num_layers,
)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print(f"Using device: {device}")
print(f"Loaded model version: {MODEL_DIR.name}")
print(f"Tokenizer type: {tokenizer.TOKENIZER_TYPE}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Inferred num_layers: {num_layers}")


Using device: cuda
Loaded model version: model_15
Tokenizer type: character_level_bpe
Tokenizer vocab size: 512
Inferred num_layers: 10


In [22]:
def prepare_prompt(prompt: str) -> str:
    token_ids = tokenizer.encode(prompt)
    if not token_ids:
        raise ValueError("Prompt must contain at least one known token")
    if len(token_ids) > MAX_LEN:
        token_ids = token_ids[-MAX_LEN:]
        prompt = tokenizer.decode(token_ids)
    return prompt


def sample_next_token(prompt: str, temperature: float = 1.0) -> int:
    prompt = prepare_prompt(prompt)
    input_ids = tokenizer.encode(prompt)
    input_tensor = torch.tensor([input_ids], device=device)

    with torch.no_grad():
        output = model(input_tensor, input_tensor)
        logits = output[0, -1, :]

        if temperature <= 0:
            return int(torch.argmax(logits).item())

        scaled_logits = logits / temperature
        probs = torch.softmax(scaled_logits, dim=-1)
        return int(torch.multinomial(probs, num_samples=1).item())


def generate_text(prompt: str, max_new_tokens: int = 50, temperature: float = 1.0) -> str:
    text = prepare_prompt(prompt)
    for _ in range(max_new_tokens):
        next_token_id = sample_next_token(text, temperature=temperature)
        next_piece = tokenizer.decode([next_token_id])
        text += next_piece
    return text

In [25]:
prompt = "It is difficult to "
temperature = 0.1
max_new_tokens = 50

completion = generate_text(prompt, max_new_tokens=max_new_tokens, temperature=temperature)
print(f"Prompt:      {prompt!r}")
print(f"Temperature: {temperature}")
print(f"Completion:  {completion}")


Prompt:      'It is difficult to '
Temperature: 0.1
Completion:  It is difficult to   s       s s           ess       e  ae ee     e  
